In [ ]:
# @title Setup Env
!pip install -q transformers datasets accelerate bitsandbytes sentence-transformers chromadb \
                langchain langchain-huggingface langchain-chroma langchain-community langchain-ollama \
                --upgrade

# 2-stage IR with `langchain_classic` compressor/reranker

After many days combing through the LangChain docs, outdated tutorials, and mutually contradicting LLMs, I figured why I was having so much trouble adding a second stage my base information retrieval pipeline.

In `langchain > 1.x` a lot has changed from the older versions, including the setup of a HF cross-encoder reranker. Some of the classes used in "the old way" now are part of `langchain_classic`, which I guess means they are deprecated until further notice?

Anyway, I still wanted to give it a shot at assembling the pipeline myself from the sources I found (and extensive help from LLMs), even though it's probably (definitely) no longer the recommended way to go with newer LangChain versions.

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors.cross_encoder_rerank import CrossEncoderReranker
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever

from sentence_transformers import CrossEncoder
from transformers import AutoTokenizer

import torch
import hashlib
import os

In [ ]:
# @title Load pretrained embedder model


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device {device}.")
model_kwargs = {'device': device, # Use 'cuda:0' to specify a GPU, if multiple available
                # 'device_map': 'auto' # Use this to automatically spread the model across multiple GPUs, or
                #       to offload parts of the model from the GPU to the CPU if it's too big for VRAM. DON'T SET BOTH
                }

model_name = "PORTULAN/serafim-100m-portuguese-pt-sentence-encoder"
embedding_model = HuggingFaceEmbeddings(model_name=model_name,
                                        model_kwargs=model_kwargs)
# Change max length for longer chunks (BERT-style models support a max sequence len of 512)
embedding_model._client.max_seq_length = 512

# Load tokenizer separately to use `RecursiveCharacterTextSplitter.from_huggingface_tokenizer`
embedding_tokenizer = AutoTokenizer.from_pretrained(model_name)

embedding_model

In [ ]:
# @title Instantiate and populate vector DB with chunked dummy documents


print("Starting Chroma...")
# Setup ChromaDB
vector_store = Chroma(
    collection_name="my_collection",
    embedding_function=embedding_model, # Embeddings are calculated on the fly
    persist_directory="./chroma_langchain_db", # Save data locally, or use host=[...] to connect to an existing ChromaDB server
    # host="localhost",
)

# Download "database"
!curl -o glossary.md https://raw.githubusercontent.com/GabrielvanderSchmidt/SENAC-DLGlossary/refs/heads/main/glossary.md

print("Doc-in' n' chunkin'...")
documents = [file for file in os.listdir(".") if os.path.isfile(file)]
documents_text = []
documents_meta = []
i = 0
for doc in documents:
    with open(doc, "r") as file:
        documents_text.append(file.read())
        documents_meta.append({"document_id" : str(i),
                               "tags" : ["learning", "AI"],
                               "source" : doc})
        i = i+1

# Convert strings to Document objects
documents = [Document(page_content=text, metadata=meta) for text, meta in zip(documents_text, documents_meta)]

# Chunk docs
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=embedding_tokenizer, # Load tokenizer to calculate chunk length
    chunk_size=200, # Chunk size (tokens) -- must be smaller than embedder sequence length
    chunk_overlap=30, # Chunk overlap (tokens) so context isn't lost at the "edges" of chunks
    add_start_index=True, # (Try to) Track index in original document
)

all_splits = text_splitter.split_documents(documents)
print(f"Documents split in {len(all_splits)} chunks.")

# Ids can be generated beforehand, or created by the `add_documents` call
# Let's generate them with our hashing function
def generate_ids(chunks):
    text_chunk = [chunk.page_content for chunk in chunks]
    encoded_chunks = [chunk.encode("utf-8") for chunk in text_chunk]
    hashes = [hashlib.md5(chunk).hexdigest() for chunk in encoded_chunks]
    return hashes

print("Embedding and pushing to vector DB...")
ids = generate_ids(all_splits)
vector_store.add_documents(documents=all_splits, ids=ids) # Outputs added ids

# Check chunking output
vector_store.get(ids=ids[0])

In [ ]:
# @title Load pretrained cross-encoder model


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device {device}.")
model_kwargs = {'device': device, # Use 'cuda:0' to specify a GPU, if multiple available
                # 'device_map': 'auto' # Use this to automatically spread the model across multiple GPUs, or to offload
                #        parts of the model from the GPU to the CPU if it's too big for VRAM. DON'T SET BOTH
                }

# model_name = "PORTULAN/albertina-1b5-portuguese-ptbr-encoder" # Final model we will want to use
model_name =  "cross-encoder/ms-marco-MiniLM-L-6-v2" # Placeholder model for now
cross_encoder_model = HuggingFaceCrossEncoder(model_name=model_name,
                                              model_kwargs=model_kwargs)
cross_encoder_model

In [ ]:
# @title Setup two-stage retrieval pipeline


# Set vector DB + embedding model as a retriever (1st stage)
base_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 20} # Retrieve a massive candidate pool for subsequent reranking
) # In this stage, we maximize recall (minimize false negatives - important context left out)

# Initialize reranking object (2nd stage)
reranker = CrossEncoderReranker(
    model=cross_encoder_model,
    top_n=3 # Retain only the 3 most strictly relevant chunks
) # In this stage, we maximize precision (minimize false positives - trash let in)

# Construct the unified two-stage retrieval pipeline
compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=base_retriever
)

for i, doc in enumerate(compression_retriever.invoke("Qual é o componente mais fundamental de redes neurais?")):
    print(f"Top {i+1} match:")
    print(doc.page_content)